<a href="https://colab.research.google.com/github/amartinsmg/classification-of-medical-images-using-cnn/blob/main/notebooks/experiment_runner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Executor de experimentos

- Usado para a execução de cada experimento no ambiente do Google Colab.
- Cada experimento foi realizado em três rodadas com seeds diferentes
- Os resultados de cada experimento foram salvas em arquivos no Google Drive e num banco de dados

Verificação do ambiente de execução

In [1]:
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices("GPU"))

!nvidia-smi

2.20.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Sat May  9 19:38:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   68C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                       

Clonagem do repositório atualizado no GitHub

In [2]:
%cd /content
!git clone https://github.com/amartinsmg/classification-of-medical-images-using-cnn.git
%cd /content/classification-of-medical-images-using-cnn

/content
fatal: destination path 'classification-of-medical-images-using-cnn' already exists and is not an empty directory.
/content/classification-of-medical-images-using-cnn


Montagem do Google Drive

In [3]:
from google.colab import drive

drive.mount("/content/drive")
BASE_PATH = "/content/drive/MyDrive/classification-of-medical-images-using-cnn/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Configuração dos parâmetros do experimento

In [4]:
EXPERIMENT_NAME = "efficientnet-final-2"
BASE_MODEL = "efficientnet"
NORMALIZATION = "preprocess-input"
DATA_AUG = True
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
CLASS_WEIGHT = True
LEARNING_RATE = 0.001
EPOCHS = 10
THRESHOLD = 0.5
seeds = [42, 123, 999]

In [5]:
RESULTS_RELATIVE_PATH = "results/" + EXPERIMENT_NAME
RESULTS_LOCAL_PATH = BASE_PATH + "/" + RESULTS_RELATIVE_PATH
MODELS_RELATIVE_PATH = "models/" + EXPERIMENT_NAME
MODELS_LOCAL_PATH = BASE_PATH + "/" + MODELS_RELATIVE_PATH

Criação da engine do banco de dados
- Foi usado um banco de dados sqlite hospedado também no Google Drive

In [6]:
from src.db import insert_run, get_engine

DB_LOCAL_PATH = f"{BASE_PATH}/experiments.db"
DB_PATH = f"sqlite:///{DB_LOCAL_PATH}"

engine = get_engine(DB_PATH)

Realização das três rodadas do experimento com:
- Treinamento do modelo
- Teste do modelo treinado

In [7]:
from src.train import train_pipeline
from src.test import test_pipeline

configs = []

for i in range(3):
    run_id = i + 1

    config = train_pipeline(
        base_dir=BASE_PATH,
        experiment_name=EXPERIMENT_NAME,
        run_id=run_id,
        normalization=NORMALIZATION,
        base_model=BASE_MODEL,
        data_augmentation=DATA_AUG,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        class_weights=CLASS_WEIGHT,
        learning_rate=LEARNING_RATE,
        epochs=EPOCHS,
        seed=seeds[i],
        versioning_models=True,
    )
    configs.append(config)

    _ = test_pipeline(
        base_dir=BASE_PATH,
        experiment_name=EXPERIMENT_NAME,
        run_id=run_id,
        normalization=NORMALIZATION,
        base_model=BASE_MODEL,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        versioning_models=True,
        save_predicitions=True,
    )

Found 5216 files belonging to 2 classes.
Found 16 files belonging to 2 classes.
Epoch 1/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 50s 124ms/step - AUC: 0.9713 - accuracy: 0.9055 - loss: 0.2143 - val_AUC: 1.0000 - val_accuracy: 1.0000 - val_loss: 0.1166
Epoch 2/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 6s 36ms/step - AUC: 0.9880 - accuracy: 0.9425 - loss: 0.1368 - val_AUC: 1.0000 - val_accuracy: 1.0000 - val_loss: 0.0592
Epoch 3/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 6s 38ms/step - AUC: 0.9928 - accuracy: 0.9578 - loss: 0.1031 - val_AUC: 1.0000 - val_accuracy: 1.0000 - val_loss: 0.0611
Epoch 4/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 6s 36ms/step - AUC: 0.9930 - accuracy: 0.9594 - loss: 0.1026 - val_AUC: 1.0000 - val_accuracy: 1.0000 - val_loss: 0.0475
Epoch 5/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 6s 38ms/step - AUC: 0.9934 - accuracy: 0.9613 - loss: 0.0966 - val_AUC: 1.0000 - val_accuracy: 1.0000 - val_loss: 0.0530
Epoch 6/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 6s 37ms/step - AUC: 0.9949 - accuracy: 0.9630 - loss: 0.0850 - val_AUC: 1

Calcula o threshold ideal

In [8]:
from src.utils import optimal_threshold

THRESHOLD = optimal_threshold(RESULTS_LOCAL_PATH)

print(f"Optimal threshold: {THRESHOLD}")

Optimal threshold: 0.9202832


In [9]:
for i in range(3):
    run_id = i + 1

    metrics = test_pipeline(
      base_dir=BASE_PATH,
      experiment_name=EXPERIMENT_NAME,
      run_id=run_id,
      normalization=NORMALIZATION,
      base_model=BASE_MODEL,
      image_size=IMAGE_SIZE,
      batch_size=BATCH_SIZE,
      threshold=THRESHOLD,
      versioning_models=True,
    )

    insert_run(
        engine=engine,
        experiment=EXPERIMENT_NAME,
        run_name=f"run-{run_id}",
        config=configs[i],
        metrics=metrics,
    )

Found 624 files belonging to 2 classes.
20/20 ━━━━━━━━━━━━━━━━━━━━ 15s 455ms/step

 Experiment Summary (efficientnet-final-2: run 1)
+----------------------+------------+-------------+----------+------------+---------------+-----------+
|   Decision Threshold |   Accuracy |   Precision |   Recall |   F1-Score |   Specificity |   AUC ROC |
+======================+============+=============+==========+============+===============+===========+
|             0.920283 |   0.899038 |    0.907731 | 0.933333 |   0.920354 |       0.84188 |  0.962404 |
+----------------------+------------+-------------+----------+------------+---------------+-----------+
Test completed. Metrics saved.
Run 'run-1' of experiment 'efficientnet-final-2' inserted.
Found 624 files belonging to 2 classes.
20/20 ━━━━━━━━━━━━━━━━━━━━ 16s 453ms/step

 Experiment Summary (efficientnet-final-2: run 2)
+----------------------+------------+-------------+----------+------------+---------------+-----------+
|   Decision Thresho

Instala DagsHub para versionamento dos resultados dos experimentos
- O DagsHub é uma plataforma de versionamento de dados e modelos, similar ao GitHub, mas focada em projetos de ciência de dados e machine learning
- Loga no DagsHub com Secrets do Google Colab
- Faz upload da pasta com os modelos treinados no último experimento

In [10]:
%pip install -q dvc dagshub

import dagshub
from google.colab import userdata

dagshub.auth.add_app_token(token=userdata.get("DAGSHUB_TOKEN"))

dagshub.upload_files(
    "amartinsmg/classification-of-medical-images-using-cnn",
    local_path=MODELS_LOCAL_PATH,
    remote_path=MODELS_RELATIVE_PATH,
)

Accessing as amartinsmg

Output()

Directory upload complete, uploaded 6 files to 
https://dagshub.com/amartinsmg/classification-of-medical-images-using-cnn/src/main/models%2Fefficientnet-final-2

Faz upload da pasta com os resultados do último experimento feito

In [11]:
dagshub.upload_files(
    "amartinsmg/classification-of-medical-images-using-cnn",
    local_path=RESULTS_LOCAL_PATH,
    remote_path=RESULTS_RELATIVE_PATH,
)

Output()

Directory upload complete, uploaded 12 files to 
https://dagshub.com/amartinsmg/classification-of-medical-images-using-cnn/src/main/results%2Fefficientnet-final-2

Copia o banco de dados do Drive para o DagsHub

In [12]:
dagshub.upload_files(
    "amartinsmg/classification-of-medical-images-using-cnn",
    local_path=DB_LOCAL_PATH,
    remote_path="experiments.db",
    force=True
)

Uploading files (1) to "amartinsmg/classification-of-medical-images-using-cnn"...

Upload finished successfully!